# Lektion 11 - Model Context Protocol (MCP)

Das **Model Context Protocol (MCP)** ist ein offener Standard, der es Agenten ermöglicht, zur Laufzeit dynamisch Werkzeuge, Ressourcen und Datenquellen zu entdecken und zu nutzen. Anstatt Werkzeuge fest in einen Agenten zu kodieren, erlaubt MCP Agenten, sich mit externen Servern zu verbinden, die Fähigkeiten auf Anfrage bereitstellen.

In dieser Lektion lernen Sie:
- Was MCP ist und warum es für Agentensysteme wichtig ist
- Wie die MCP Client-Server-Architektur funktioniert
- Wie man Agenten baut, die MCP-artige Werkzeugerkennung verwenden


## Einrichtung

**Voraussetzungen:**
- Microsoft Foundry-Projekt mit bereitgestelltem Modell
- Führen Sie `az login` zur Authentifizierung aus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Was ist das Model Context Protocol (MCP)?

MCP definiert eine standardisierte Methode für KI-Agenten, externe Werkzeuge und Datenquellen zu entdecken und mit ihnen zu interagieren:

- **MCP Server**: Stellt Werkzeuge, Ressourcen und Eingabeaufforderungen über ein Standardprotokoll bereit
- **MCP Client**: Die Agenten-Laufzeitumgebung, die sich mit Servern verbindet und verfügbare Funktionen entdeckt
- **Dynamische Entdeckung**: Agenten benötigen keine fest codierten Werkzeuge — sie entdecken zur Laufzeit, was verfügbar ist

Dies ist leistungsstark für den Aufbau erweiterbarer Agentensysteme, bei denen neue Fähigkeiten hinzugefügt werden können, ohne den Agentencode zu ändern.


## Wie MCP funktioniert

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Der Agent (MCP-Client) verbindet sich mit einem MCP-Server
2. Der Server antwortet mit einer Liste verfügbarer Tools und deren Schemata
3. Der Agent kann dann während seines Denkprozesses jedes entdeckte Tool aufrufen
4. Ergebnisse fließen über dasselbe Protokoll zurück


## Simulation der MCP-Tool-Erkennung

Da ein echter MCP-Server einen laufenden Serverprozess benötigt, demonstrieren wir das Muster mit `@tool`-Funktionen, die simulieren, was ein mit MCP verbundener Unterkunftsdienst bereitstellen würde.

In der Produktion würden diese Tools dynamisch von einem MCP-Server entdeckt und nicht lokal definiert werden.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Aufbau eines Agenten mit MCP-ähnlichen Werkzeugen


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP in der Produktion

In einer Produktionsumgebung ermöglicht MCP leistungsstarke Muster:

- **Dynamische Werkzeugerkennung**: Agenten verbinden sich mit MCP-Servern und entdecken Werkzeuge zur Laufzeit
- **Entkoppelte Architektur**: Werkzeuganbieter können unabhängig vom Agenten aktualisieren
- **Organisationenübergreifende Freigabe**: Teams können Fähigkeiten über MCP-Server bereitstellen, die von jedem Agenten genutzt werden können
- **Unterstützung des Microsoft Agent Framework**: MAF hat integrierte MCP-Client-Unterstützung über die `mcp`-Integration

Um einen echten MCP-Server mit MAF zu verwenden, verbinden Sie sich über `hosted_mcp_tool()` oder die MCP-Client-Integration.

**Mehr erfahren:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
